# 01 - Advanced EDA (Time Series + Cohorts + Anomalies)

This notebook creates the point-in-time training dataset and performs advanced EDA.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.pipelines.churn_common import create_db_engine, fetch_churn_feature_frame, generate_snapshot_schedule
from ml.pipelines.churn_advanced_eda import (
    compute_snapshot_metrics,
    compute_missingness,
    compute_correlations,
    compute_cohort_retention,
    compute_gap_anomalies,
)

pd.set_option('display.max_columns', 120)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATA_DIR = ROOT / 'ml' / 'data'
REPORT_DIR = DATA_DIR / 'reports' / 'churn'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

engine = create_db_engine()


In [ ]:
# Cell 1: Build snapshot schedule
snapshots = generate_snapshot_schedule(
    engine,
    history_days=180,
    label_days=30,
    snapshots=12,
    spacing_days=14,
)

print('Snapshot count:', len(snapshots))
print([s.date().isoformat() for s in snapshots])


In [ ]:
# Cell 2: Build point-in-time feature dataset
frames = []
for s in snapshots:
    frame = fetch_churn_feature_frame(
        engine,
        snapshot_ts=s,
        history_days=180,
        label_days=30,
        include_label=True,
    )
    if frame.empty:
        continue
    frames.append(frame)
    print(f'snapshot={s.date().isoformat()} rows={len(frame)} target_rate={frame["will_order_next_30d"].mean():.4f}')

assert frames, 'No rows produced from snapshots.'
dataset = pd.concat(frames, ignore_index=True)
dataset['snapshot_date'] = pd.to_datetime(dataset['snapshot_date'], utc=True)
dataset['user_id'] = dataset['user_id'].astype(str)
dataset = dataset.drop_duplicates(subset=['user_id', 'snapshot_date']).sort_values(['snapshot_date', 'user_id'])

dataset_path = DATA_DIR / 'churn_training_dataset.csv'
dataset.to_csv(dataset_path, index=False)
print('Saved dataset:', dataset_path)
print('Dataset shape:', dataset.shape)


In [ ]:
# Cell 3: Core profile + target balance
print('Rows:', len(dataset))
print('Users:', dataset['user_id'].nunique())
print('Snapshots:', dataset['snapshot_date'].nunique())
print('Target positive rate:', dataset['will_order_next_30d'].mean())
display(dataset.head(5))


In [ ]:
# Cell 4: Snapshot-level trends
snapshot_metrics = compute_snapshot_metrics(dataset)
display(snapshot_metrics)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(snapshot_metrics['snapshot_date'], snapshot_metrics['target_rate'], label='target_rate')
ax.plot(snapshot_metrics['snapshot_date'], snapshot_metrics['avg_orders_30d'], label='avg_orders_30d')
ax.set_title('Snapshot Trends')
ax.grid(alpha=0.25)
ax.legend()
plt.show()


In [ ]:
# Cell 5: Missingness and correlations
missingness = compute_missingness(dataset)
correlations = compute_correlations(dataset)

display(missingness.head(20))
display(correlations.head(20))


In [ ]:
# Cell 6: Cohort retention and churn-anomaly view
retention = compute_cohort_retention(dataset)
gap_anomalies = compute_gap_anomalies(dataset)

display(retention.head(20))
display(gap_anomalies.head(20))


In [ ]:
# Cell 7: Save EDA artifacts
snapshot_metrics.to_csv(REPORT_DIR / 'snapshot_metrics.csv', index=False)
missingness.to_csv(REPORT_DIR / 'missingness.csv', index=False)
correlations.to_csv(REPORT_DIR / 'feature_target_correlations.csv', index=False)
retention.to_csv(REPORT_DIR / 'cohort_retention.csv', index=False)
gap_anomalies.to_csv(REPORT_DIR / 'gap_anomalies.csv', index=False)

report_text = f"""
# Advanced EDA Summary

- generated_at_utc: {datetime.now(UTC).isoformat()}
- rows: {len(dataset)}
- users: {dataset['user_id'].nunique()}
- snapshots: {dataset['snapshot_date'].nunique()}
- target_positive_rate: {dataset['will_order_next_30d'].mean():.4f}
"""
(REPORT_DIR / 'eda_report.md').write_text(report_text, encoding='utf-8')
print('Saved EDA outputs to:', REPORT_DIR)


## Next Notebook

Proceed to `02_feature_engineering_and_selection.ipynb`.
